Simulated Annealing Experiments

In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
from datetime import datetime

# Parameters
results_path = f'data/simulated_annealing/summary_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}.npy'
brute_force_data_directory = './data/brute_force/test'

num_runs = 2

# I think init and end temperatures should be chosen dependent on the delta of the energy,
# i.e. delta = new_loss - old_loss (in the exponent)
# larger delta -> larger temperature (and conversely smaller beta)
init_temp = 1000
end_temp = 1
max_temp_iterations = 101

In [16]:
from pathlib import Path
import numpy as np
import pandas as pd
import re

data_files = [p for p in Path(brute_force_data_directory).glob("*.csv") if p.is_file()]
candidate_dims = [(np.log2(int(re.search(string=str(p.name), pattern=r'_(\d*)_candidates').group(1))), p) for p in data_files]

df = pd.DataFrame(data=candidate_dims, columns=['dims','paths'])
df = df.groupby('dims')['paths'].agg(list).reset_index()

In [17]:
problem_sizes = df['dims'].to_numpy()
brute_force_data_paths = np.array(df['paths'].to_list(), dtype=object)

In [18]:
problem_sizes.shape

(4,)

In [19]:
brute_force_data_paths.shape

(4, 100)

In [20]:
import pandas as pd
import numpy as np
brute_force_data = np.vectorize(pd.read_csv)(brute_force_data_paths)
loss_spaces = [np.array([d['loss'] for d in data]) for data in brute_force_data] # needs to be list because is inhomogeneous

In [21]:
from cl_optimizer import SimulatedAnnealing

def create_simulated_annealing_instances(x: np.ndarray):
    return np.array([[
        SimulatedAnnealing(
            lookup_table=data_path,
            bounds=int(x[0]),
        ) for _ in range(num_runs)
    ] for data_path in x[1:]])

sim_annss = np.apply_along_axis(func1d=create_simulated_annealing_instances, arr=np.column_stack((problem_sizes, brute_force_data)), axis=1) # a sim_ann per problem size and instance

In [22]:
sim_annss.shape

(4, 100, 2)

In [23]:
from collections import defaultdict
from joblib import Parallel, delayed
from parallelization import construct_parallelization_arg_list, get_execute_sim_ann

resultsss = []
for i, sim_ann_list in enumerate(sim_annss):
    resultss = []
    for j, sim_anns in enumerate(sim_ann_list):
        arg_list = construct_parallelization_arg_list(sim_anns, depth=max_temp_iterations, step=10, batch_size=8)

        print(f"Problem size: {i + 1}/{int(problem_sizes[-1])} - instances: {j + 1}/{len(sim_ann_list)}")
        print(f"Preparing to execute {len(arg_list)} in parallel.")
        results = Parallel(n_jobs=len(arg_list), verbose=11)(
            delayed(get_execute_sim_ann(init_temp, end_temp))(s_id, sim_ann, temp_iterations) for (s_id, sim_ann), temp_iterations in arg_list)

        groups = defaultdict(list)
        for key, arr in results:
            groups[key].append(arr)

        # Column stack all arrays with the same key
        results = np.array([np.column_stack(groups[k]) for k in sorted(groups)])

        resultss.append(results)
    resultsss.append(resultss)

Problem size: 1/4 - instances: 1/100
Preparing to execute 4 in parallel.


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.4s remaining:    0.4s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.5s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.05695939064025879s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.05363798141479492s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: 

Problem size: 1/4 - instances: 2/100
Preparing to execute 4 in parallel.
Problem size: 1/4 - instances: 3/100
Preparing to execute 4 in parallel.
Problem size: 1/4 - instances: 4/100
Preparing to execute 4 in parallel.


[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.051177978515625s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.05769991874694824s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.0504908561706543s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Para

Problem size: 1/4 - instances: 5/100
Preparing to execute 4 in parallel.
Problem size: 1/4 - instances: 6/100
Preparing to execute 4 in parallel.
Problem size: 1/4 - instances: 7/100
Preparing to execute 4 in parallel.


...

[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.050093650817871094s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.055776119232177734s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch co

Problem size: 4/4 - instances: 95/100
Preparing to execute 4 in parallel.
Problem size: 4/4 - instances: 96/100
Preparing to execute 4 in parallel.
Problem size: 4/4 - instances: 97/100
Preparing to execute 4 in parallel.


[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.05213427543640137s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.05507063865661621s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch comp

Problem size: 4/4 - instances: 98/100
Preparing to execute 4 in parallel.
Problem size: 4/4 - instances: 99/100
Preparing to execute 4 in parallel.
Problem size: 4/4 - instances: 100/100
Preparing to execute 4 in parallel.


[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished


In [24]:
result_tensor = np.stack(resultsss, axis=0).squeeze()

In [25]:
indices = np.argsort(result_tensor[..., 0], axis=3) # sort by temp. iterations (parallelization does not guarantee order) along runs (axis=3)
indices_expanded = np.expand_dims(indices, axis=-1)
result_tensor = np.take_along_axis(result_tensor, indices_expanded, axis=3)

In [26]:
result_tensor.shape # should be a 5-tuple: (n_problem_sizes, n_instances_per_problem_size, n_runs_per_instance, n_temp_iter_steps, 2)

(4, 100, 2, 11, 2)

In [27]:
loss_tensor = result_tensor[:, :, :, :, 1] # loss has index 1 in last dim

In [28]:
np.save(results_path, result_tensor)